# Module 6: AI workforce data cleaning

This notebook provides a reviewable path through the same full-dataset checks implemented in `clean_data.py`. The project compares observed AI exposure across occupations and task penetration while treating exposure as workflow evidence rather than proof of job replacement.

In [ ]:
from pathlib import Path
from collections import Counter
import csv
import hashlib
import json
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / 'clean_data.py').exists():
    raise FileNotFoundError('Open this notebook with the project folder as the working directory.')
print(f'Project folder: {ROOT}')
print(f'Python: {sys.version.split()[0]}')

## Inspect the unchanged inputs

The source files are identified by schema and SHA-256 hash. Exact duplicates and missing required fields are counted before any output is written.

In [ ]:
def inspect_csv(filename):
    path = ROOT / filename
    with path.open(encoding='utf-8-sig', newline='') as stream:
        reader = csv.DictReader(stream)
        rows = list(reader)
        columns = reader.fieldnames
    signatures = [tuple(row[column] for column in columns) for row in rows]
    return {
        'file': filename,
        'sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
        'columns': columns,
        'rows': len(rows),
        'missing': {column: sum(not row[column].strip() for row in rows) for column in columns},
        'exact_duplicate_excess': len(rows) - len(set(signatures)),
    }

source_profiles = [inspect_csv('job_exposure.csv'), inspect_csv('task_penetration.csv')]
for profile in source_profiles:
    print(json.dumps(profile, indent=2))

## Apply the declared rules

The rules preserve identifiers and source wording, validate scores as finite decimals in `[0, 1]`, stop on missing or malformed values, and retain one occurrence of each exact row. Case and punctuation remain meaningful, so the `Web` and `web` task variants stay separate.

In [ ]:
completed = subprocess.run(
    [sys.executable, str(ROOT / 'clean_data.py')],
    cwd=ROOT,
    check=True,
    capture_output=True,
    text=True,
)
run_report = json.loads(completed.stdout)
print('Cleaning completed from the original CSV files.')
for dataset, evidence in run_report.items():
    print(f"{dataset}: {evidence['rows_before']:,} -> {evidence['rows_after']:,} rows")

## Reconcile the full dataset

In [ ]:
validation = json.loads((ROOT / 'cleaned' / 'validation.json').read_text(encoding='utf-8'))
for dataset, evidence in validation.items():
    print(f'\n{dataset}')
    print(f"  rows: {evidence['rows_before']:,} -> {evidence['rows_after']:,}")
    print(f"  exact duplicate excess before: {evidence['exact_duplicate_excess_before']}")
    print(f"  missing before: {evidence['missing_before']}")
    print(f"  missing after: {evidence['missing_after']}")
    print(f"  score range after: {evidence['metrics_after']['min']} to {evidence['metrics_after']['max']}")
    print(f"  score sum removed: {evidence['score_sum_removed']}")

assert validation['job_exposure']['rows_after'] == 756
assert validation['task_penetration']['rows_after'] == 17992
assert validation['task_penetration']['removed_rows'] == 6
assert validation['task_penetration']['score_sum_removed'] == '4.3578'
print('\nCount and score reconciliation passed.')

## Verify five difficult records

In [ ]:
record_checks = json.loads((ROOT / 'cleaned' / 'record_checks.json').read_text(encoding='utf-8'))
assert len(record_checks) == 5
assert all(check['matches'] for check in record_checks)
for number, check in enumerate(record_checks, 1):
    key = check['original'].get('occ_code', check['original'].get('task'))
    print(f"{number}. {check['dataset']}: {key} | source count={check['original_count']} | match={check['matches']}")
print('All five record checks passed.')

## Remaining limitations

The task table has no occupation code or task identifier, so the files cannot be safely merged and similar task descriptions cannot be consolidated authoritatively. Exact-row deduplication supports distinct-description analysis but changes occurrence weighting. Exposure and penetration do not establish causation, displacement, implementation ROI, or timing.